# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*
## Feature vector construction

The model uses only information available before the refresh decision.

Selected features represent:
- current content performance (traffic, CTR, ranking)
- search demand
- content freshness
- page characteristics

Categorical features are converted into numeric form.
Missing numerical values are filled using median values because they are less sensitive to extreme values than averages.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Features available before prediction
feature_cols = [
    "impressions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "position_tier"
]

# Create feature dataframe
X = df[feature_cols].copy()

# Handle categorical feature
X = pd.get_dummies(
    X,
    columns=["position_tier"],
    dummy_na=True
)

# Fill missing numerical values
X = X.fillna(X.median())

# Check result
X.head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
## Feature explanations

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| impressions_90d | Number of impressions received in the last 90 days | Filled with median | Yes |
| search_volume | Estimated search demand for the topic | Filled with median | Yes |
| ctr | Click-through rate from search impressions | Filled with median | Yes |
| avg_position | Average search ranking position | Filled with median | Yes |
| content_age_days | Age of the content page | Filled with median | Yes |
| days_since_last_update | Time since content was last updated | Filled with median | Yes |
| word_count | Length of the content | Filled with median | Yes |
| position_tier | Ranking category, treated as categorical | Converted using one-hot encoding | Yes |

All selected features describe the state of the page before deciding whether it needs refresh.
No future information or outcome labels are included.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
## Leakage checks

Feature leakage happens when information that would not be available at prediction time is included in the model.

I checked for:
- direct label leakage
- features with suspiciously strong relationships with the target
- fields that represent future information

The target column (`is_declining_label`) is excluded from the feature vector and is only used as the prediction target.

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Check 1: Make sure identifier columns are not in the feature vector
excluded_fields = [
    "client_hash_id",
    "content_hash_id"
]

print("Identifier leakage check:")
for field in excluded_fields:
    print(f"{field} included in features:", field in X.columns)


# Check 2: Look for possible time-related leakage fields
date_columns = [
    col for col in df.columns
    if "date" in col.lower()
    or "time" in col.lower()
    or "days" in col.lower()
]

print("\nPossible time-related columns:")
print(date_columns)


# Check 3: Check whether target column accidentally entered features
target_column = "is_declining_label"

print("\nTarget leakage check:")
print(
    f"{target_column} included in features:",
    target_column in X.columns
)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
## Excluded fields

| Field | Reason for exclusion |
|---|---|
| is_declining_label | This is the target variable, not a feature. Including it would leak the answer to the model. |
| client_hash_id | Identifier only; does not describe page quality or user behavior. |
| content_hash_id | Identifier only; does not provide predictive information. |
| report_date | Represents when the observation was collected and could create time leakage. |

Only information available before the prediction decision is included.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.